# Task 12: MongoDB Analytics
## COVID-19 Twitter Data Analysis using Python + MongoDB + SciPy

**Technologies:** MongoDB | PyMongo | NLTK | SciPy | Pandas | Google Colab

### Key Queries
- **Query i** — Total `favorite_count` grouped by `place_type`
- **Query ii** — Total `retweet_count` grouped by `country_code`
- **Query iii** — Top 10 frequent words after Lemmatization + Stop Word Removal

> ⚡ Run each cell in order using **Shift + Enter**

---
## STEP 1 — Install & Start MongoDB in Colab

In [ ]:
# Cell 1 — Install & Start MongoDB
# Install MongoDB in Google Colab
!wget -qO - https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-6.0.6.tgz | tar -xz
!rm -rf /usr/local/mongodb
!mv mongodb-linux-x86_64-ubuntu2204-6.0.6 /usr/local/mongodb
!mkdir -p /data/db

# Start MongoDB in background
!nohup /usr/local/mongodb/bin/mongod --dbpath /data/db --bind_ip 127.0.0.1 --port 27017 > /tmp/mongo.log 2>&1 &

# Wait 3 seconds for MongoDB to start
!sleep 3
print('MongoDB started successfully')

---
## STEP 2 — Install Python Libraries

In [ ]:
# Cell 2 — Install all required Python libraries
!pip install pymongo scipy pandas matplotlib gdown nltk wordcloud

---
## STEP 3 — Download Dataset from Google Drive

In [ ]:
# Cell 3 — Download COVID-19 Twitter dataset
import gdown

# Download COVID-19 dataset from Google Drive
file_id = '1o1RfInuyj3FfTIqK3Eoeg2ieUby2DSUw'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'covid_dataset.csv'
gdown.download(url, output, quiet=False)

print('Dataset downloaded successfully')

---
## STEP 4 — Load Dataset into Pandas

In [ ]:
# Cell 4 — Load CSV and preview data
import pandas as pd
import json

# Load the CSV file
df = pd.read_csv('covid_dataset.csv')

print('Dataset loaded:', df.shape)
print('Columns:', df.columns.tolist())
print(df.head())

---
## STEP 5 — Insert Data into MongoDB

In [ ]:
# Cell 5 — Connect to MongoDB and insert tweets
from pymongo import MongoClient

# Select columns and take first 100 rows as sample
sample_data = df[['id', 'created_at', 'original_text',
                   'favorite_count', 'retweet_count',
                   'sentiment']].head(100).to_dict(orient='records')

# Save as JSON file
with open('covid19_tweets.json', 'w') as f:
    json.dump(sample_data, f, indent=4)
print('Sample saved as covid19_tweets.json')

# Connect to MongoDB
client = MongoClient('mongodb://127.0.0.1:27017/')
db = client['twitter_db']
collection = db['covid19_tweets']

# Clear old data and insert fresh
collection.delete_many({})
collection.insert_many(sample_data)

print('Tweets inserted into MongoDB')
print('Total documents:', collection.count_documents({}))

---
## STEP 6 — SciPy Statistical Analysis

In [ ]:
# Cell 6 — Statistics using SciPy
import numpy as np
from scipy import stats

# Get values from dataset
likes         = df['favorite_count'].dropna().values
retweets      = df['retweet_count'].dropna().values
tweet_lengths = df['original_text'].dropna().apply(len).values

print('--- Statistical Analysis ---')
print('Avg tweet length :', round(np.mean(tweet_lengths), 2))
print('Avg likes        :', round(np.mean(likes), 2))
print('Avg retweets     :', round(np.mean(retweets), 2))

# T-test: Are avg likes significantly greater than 10?
t_stat, p_val = stats.ttest_1samp(likes, 10)
print('\nT-test (likes > 10)')
print('t-stat:', round(t_stat, 4))
print('p-val :', round(p_val, 4))

if p_val < 0.05 and np.mean(likes) > 10:
    print('Result: Significant — avg likes > 10')
else:
    print('Result: Not significant')

---
## STEP 7 — MongoDB Aggregation Queries

### Query i: Total favorite_count grouped by place_type

In [ ]:
# Cell 7 — Query i: Total favorite_count for each place_type
print('=== Query i: Favorite Count by Place Type ===')
print()

pipeline_i = [
    {
        '$group': {
            '_id': '$place_type',
            'total_favorite_count': { '$sum': '$favorite_count' }
        }
    },
    { '$sort': { 'total_favorite_count': -1 } }
]

results_i = list(collection.aggregate(pipeline_i))

for row in results_i:
    place = row['_id'] if row['_id'] else 'Unknown/None'
    print(f'Place Type: {place:20s}  |  Total Favorites: {row["total_favorite_count"]}')

print(f'\nTotal place_type groups found: {len(results_i)}')

### Query ii: Total retweet_count grouped by country_code

In [ ]:
# Cell 7b — Query ii: Total retweet_count for each country_code
print('=== Query ii: Retweet Count by Country Code ===')
print()

pipeline_ii = [
    {
        '$group': {
            '_id': '$country_code',
            'total_retweet_count': { '$sum': '$retweet_count' }
        }
    },
    { '$sort': { 'total_retweet_count': -1 } }
]

results_ii = list(collection.aggregate(pipeline_ii))

for row in results_ii:
    country = row['_id'] if row['_id'] else 'Unknown/None'
    print(f'Country Code: {str(country):10s}  |  Total Retweets: {row["total_retweet_count"]}')

print(f'\nTotal country_code groups found: {len(results_ii)}')

---
## STEP 8 — NLP: Top 10 Words (Query iii)

In [ ]:
# Cell 8 — Query iii: Top 10 words after lemmatization
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

# Fetch all tweet texts from MongoDB
all_tweets = collection.find({}, {'original_text': 1, '_id': 0})
text_data = ' '.join([
    doc['original_text']
    for doc in all_tweets
    if doc.get('original_text')
])

# Step 1: Clean text — remove special chars, lowercase
text_data = re.sub(r'[^a-zA-Z ]', ' ', text_data.lower())

# Step 2: Tokenize
tokens = text_data.split()

# Step 3: Remove stop words and short words
stop_words = set(stopwords.words('english'))
tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

# Step 4: Lemmatize
lemmatizer = WordNetLemmatizer()
tokens = [lemmatizer.lemmatize(w) for w in tokens]

# Step 5: Count frequency
word_counts = Counter(tokens)
top_10_words = word_counts.most_common(10)

print('=== Query iii: Top 10 Most Frequent Words ===')
print()
for rank, (word, freq) in enumerate(top_10_words, 1):
    print(f'  {rank:2}. {word:20s} — frequency: {freq}')

---
## STEP 9 — Visualizations (4 Charts)

In [ ]:
# Cell 9 — All 4 charts
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# --- Chart 1: Sentiment Distribution ---
sentiment_counts = df['sentiment'].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(sentiment_counts.index, sentiment_counts.values,
        color=['green', 'gray', 'red'])
plt.title('Chart 1: Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# --- Chart 2: Tweets Per Day ---
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
tweets_per_day = df.groupby(df['created_at'].dt.date).size()
tweets_per_day.plot(figsize=(10, 5), title='Chart 2: Tweets Per Day', color='blue')
plt.xlabel('Date')
plt.ylabel('Tweet Count')
plt.tight_layout()
plt.show()

# --- Chart 3: Top 10 Frequent Words Bar Chart ---
words, freqs = zip(*top_10_words)
plt.figure(figsize=(10, 5))
plt.bar(words, freqs, color='skyblue')
plt.title('Chart 3: Top 10 Frequent Words')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Chart 4: Word Cloud ---
wordcloud = WordCloud(
    width=800, height=400,
    background_color='white'
).generate(' '.join(tokens))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Chart 4: Word Cloud of COVID-19 Tweets')
plt.tight_layout()
plt.show()

---
## Expected Output Summary

### Query i — Favorite Count by Place Type
```
=== Query i: Favorite Count by Place Type ===
Place Type: Unknown/None         |  Total Favorites: 1.0
```

### Query ii — Retweet Count by Country Code
```
=== Query ii: Retweet Count by Country Code ===
Country Code: None       |  Total Retweets: 101034.0
```

### Query iii — Top 10 Most Frequent Words
```
=== Query iii: Top 10 Most Frequent Words ===
   1. covid                — frequency: 61
   2. http                 — frequency: 38
   3. people               — frequency: 9
   4. country              — frequency: 8
   5. coronavirus          — frequency: 7
   6. died                 — frequency: 7
   7. get                  — frequency: 6
   8. death                — frequency: 6
   9. family               — frequency: 5
  10. one                  — frequency: 5
```

---
### Cell Execution Order

| Cell | What it does | Key Output |
|------|-------------|------------|
| 1 | Install & start MongoDB | `MongoDB started successfully` |
| 2 | Install Python libraries | pymongo, nltk, scipy installed |
| 3 | Download dataset | covid_dataset.csv (~54 MB) |
| 4 | Load CSV with Pandas | DataFrame with tweet data |
| 5 | Insert 100 tweets into MongoDB | `Total documents: 100` |
| 6 | SciPy statistics | Avg likes, retweets, t-test result |
| 7 | Query i — fav by place_type | Favorite counts per place |
| 7b | Query ii — retweet by country | Retweet counts per country |
| 8 | Query iii — Top 10 words | covid, people, coronavirus... |
| 9 | 4 Visualizations | Bar charts + Word Cloud |